In [1]:
import pandas as pd
df = pd.read_csv("/content/sample_data/industrialcombenergy-2014.csv")

cols_to_drop = [
    "FACILITY_ID"
    "FACILITY_NAME",
    "UNIT_NAME",
    "UNIT_TYPE",
    "OTHER_OR_BLEND_FUEL_TYPE",
    "PRIMARY_NAICS_TITLE",
    "GROUPING"
]

cols_to_drop = [c for c in cols_to_drop if c in df.columns]

df_clean = df.drop(columns=cols_to_drop)

#COGENERATION_UNIT_EMISS_IND → 1 / 0
df_clean["COGENERATION_UNIT_EMISS_IND"] = df_clean["COGENERATION_UNIT_EMISS_IND"].map({
    "Y": 1,
    "N": 0
})

# 4) One-hot encoding
df_encoded = pd.get_dummies(df_clean, columns=["FUEL_TYPE"], prefix="fuel")

fuel_cols = [col for col in df_encoded.columns if col.startswith("fuel_")]
df_encoded[fuel_cols] = df_encoded[fuel_cols].astype(int)

output_path = "/content/sample_data/clean_industrial_energy_encoded1.csv"
df_encoded.to_csv(output_path, index=False)

output_path


'/content/sample_data/clean_industrial_energy_encoded1.csv'

In [2]:
import pandas as pd

df = pd.read_csv("/content/sample_data/clean_industrial_energy_encoded1.csv")
df.head(10)

,FACILITY_ID,FACILITY_NAME,REPORTING_YEAR,PRIMARY_NAICS_CODE,COGENERATION_UNIT_EMISS_IND,MMBtu_TOTAL,GWht_TOTAL,fuel_Agricultural Byproducts,fuel_Anthracite,fuel_Biodiesel (100%),...,fuel_Residual Fuel Oil No. 5,fuel_Residual Fuel Oil No. 6,fuel_Solid Byproducts,fuel_Special Naphtha,fuel_Subbituminous,fuel_Tires,fuel_Unfinished Oils,fuel_Used Oil,fuel_Vegetable Oil,fuel_Wood and Wood Residuals (dry basis)
0,1003826,ATLANTIC WASTE DISPOSAL INC - SUSSEX COUNTY LAND,2014,562212,0,1095.021152,0.320921,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1003826,ATLANTIC WASTE DISPOSAL INC - SUSSEX COUNTY LAND,2014,562212,0,1488.773186,0.436319,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1003826,ATLANTIC WASTE DISPOSAL INC - SUSSEX COUNTY LAND,2014,562212,0,1488.773186,0.436319,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1003826,ATLANTIC WASTE DISPOSAL INC - SUSSEX COUNTY LAND,2014,562212,0,2803.449398,0.821615,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,1005928,LARIMER COUNTY LANDFILL,2014,562212,0,847.512559,0.248383,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,1005964,GOLDEN TRIANGLE REGIONAL SOLID WASTE MANAGEMEN...,2014,562111,0,78110.236220,22.891996,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6,1007951,BRADLEY LANDFILL & RECYCLING CENTER,2014,562212,0,314.738032,0.092241,0,0,0,...,0,0,0,0,0,0,0,0,0,0
7,1007951,BRADLEY LANDFILL & RECYCLING CENTER,2014,562212,0,314.738032,0.092241,0,0,0,...,0,0,0,0,0,0,0,0,0,0
8,1007951,BRADLEY LANDFILL & RECYCLING CENTER,2014,562212,0,314.738032,0.092241,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9,1007951,BRADLEY LANDFILL & RECYCLING CENTER,2014,562212,0,314.738032,0.092241,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [3]:
df.to_csv("/content/sample_data/industrial_energy_with_predictions.csv", index=False)

In [7]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_score, recall_score, f1_score

# Loading cleaned + encoded dataset
df = pd.read_csv("/content/sample_data/clean_industrial_energy_encoded1.csv")

# Droping non-numeric or irrelevant columns
columns_to_drop = ["PRIMARY_NAICS_TITLE", "GROUPING","FACILITY_NAME"]
df = df.drop(columns=columns_to_drop, errors="ignore")

# Ensure boolean fuel columns are converted to 0/1
fuel_cols = [col for col in df.columns if col.startswith("fuel_")]
df[fuel_cols] = df[fuel_cols].astype(int)

# Select features
if "predicted_waste" in df.columns:
    df = df.drop(columns=["predicted_waste"])

X = df.copy()

# Scale the data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train Isolation Forest
iso = IsolationForest(
    n_estimators=300,
    contamination=0.05,   # 5% anomalies
    random_state=42
)

iso.fit(X_scaled)

# Raw model output: 1=normal, -1=anomaly
y_raw = iso.predict(X_scaled)

# Convert to waste labels (1=waste, 0=normal)
y_pred = np.where(y_raw == -1, 1, 0)

# Add predictions back to dataframe
df["predicted_waste"] = y_pred

# Save model + scaler
import joblib
joblib.dump(iso, "isolation_forest_model.pkl")
joblib.dump(scaler, "scaler.pkl")

df.to_csv("isolated_predictions_output.csv", index=False)

print("Model training complete")


Model training complete


In [10]:
import pandas as pd
import joblib

df = pd.read_csv("/content/sample_data/industrialcombenergy-2014.csv")
print(df.columns)

# calculating the avrage for each FULE TYPE
defaults = df.groupby("FUEL_TYPE")[["MMBtu_TOTAL", "GWht_TOTAL"]].mean()

# {fuel: {"MMBtu": x, "GWht": y}}
fuel_defaults = {}

for fuel in defaults.index:
    avg_mmbtu = defaults.loc[fuel, "MMBtu_TOTAL"]
    avg_gwht = defaults.loc[fuel, "GWht_TOTAL"]

    fuel_defaults[fuel] = {
        "MMBtu": float(avg_mmbtu),
        "GWht": float(avg_gwht)
    }

joblib.dump(fuel_defaults, "fuel_defaults.pkl")

print("fuel_defaults has been created")
fuel_defaults

Index(['FACILITY_ID', 'FACILITY_NAME', 'FUEL_TYPE', 'OTHER_OR_BLEND_FUEL_TYPE',
       'REPORTING_YEAR', 'UNIT_NAME', 'UNIT_TYPE', 'PRIMARY_NAICS_CODE',
       'PRIMARY_NAICS_TITLE', 'COGENERATION_UNIT_EMISS_IND', 'MMBtu_TOTAL',
       'GWht_TOTAL', 'GROUPING'],
      dtype='object')
fuel_defaults has been created


{'Agricultural Byproducts': {'MMBtu': 944376.159138643,
  'GWht': 276.7710863615},
 'Anthracite': {'MMBtu': 243104.22237526003, 'GWht': 71.2472663226},
 'Biodiesel (100%)': {'MMBtu': 75297.03864324445, 'GWht': 22.067523601555553},
 'Bituminous': {'MMBtu': 1543696.0706531007, 'GWht': 452.41552778925063},
 'Blast Furnace Gas': {'MMBtu': 1435462.398848424, 'GWht': 420.695168648913},
 'Butane': {'MMBtu': 300714.32246, 'GWht': 88.13122705666667},
 'Coal Coke': {'MMBtu': 392446.8146456182, 'GWht': 115.01553715016364},
 'Coke Oven Gas': {'MMBtu': 980962.1366585294, 'GWht': 287.4934459200882},
 'Distillate Fuel Oil No. 1': {'MMBtu': 10672.176428706756,
  'GWht': 3.127725997648649},
 'Distillate Fuel Oil No. 2': {'MMBtu': 30293.368284007443,
  'GWht': 8.87816614994326},
 'Distillate Fuel Oil No. 4': {'MMBtu': 47380.400952386546,
  'GWht': 13.885912849206898},
 'Ethane': {'MMBtu': 794257.1907857143, 'GWht': 232.77528069285714},
 'Ethanol (100%)': {'MMBtu': 84129.16423, 'GWht': 24.65598051},
 'Et